# ControlSift — Safe free GPU runner (Kaggle / Colab)

**Security**
- Put your Hugging Face token in platform secrets as `HF_TOKEN` only.
- Never paste a token into a cell, chat, or committed file.
- This notebook never prints the token.

**Before you start**
1. Accept the Gemma license on Hugging Face for `google/gemma-3-1b-it`.
2. Kaggle: Add-ons → Secrets → `HF_TOKEN` · Accelerator = GPU.
3. Attach the ControlSift Kaggle Dataset **or** set `REPO_URL` to your GitHub clone URL.

Keep `SMOKE = True` for the first session.

In [ ]:
# === Config (safe defaults) ===
SMOKE = True          # True = short validation run; False = full test/challenge + train
SMOKE_LIMIT = 16
REPO_URL = ""         # optional: https://github.com/<you>/controlsift.git
KAGGLE_DATASET_DIR = "/kaggle/input"  # notebook will search under here for controlsift/
WORKDIR = "/kaggle/working/controlsift"  # Colab users: change to /content/controlsift

print("SMOKE =", SMOKE)

In [ ]:
# === Locate or fetch repo (no secrets) ===
import os
import shutil
from pathlib import Path

work = Path(WORKDIR)
work.parent.mkdir(parents=True, exist_ok=True)

def find_bundled_repo(root: Path) -> Path | None:
    if not root.exists():
        return None
    for candidate in root.rglob("pyproject.toml"):
        text = candidate.read_text(encoding="utf-8", errors="ignore")
        if 'name = "controlsift"' in text or "name = 'controlsift'" in text:
            return candidate.parent
    return None

src = find_bundled_repo(Path(KAGGLE_DATASET_DIR))
if src is None and REPO_URL:
    if work.exists():
        shutil.rmtree(work)
    !git clone --depth 1 {REPO_URL} {work}
elif src is not None:
    if work.exists():
        shutil.rmtree(work)
    shutil.copytree(src, work)
    print("Copied dataset bundle from", src)
elif Path("/content/controlsift").exists():
    work = Path("/content/controlsift")
    print("Using existing /content/controlsift")
else:
    raise SystemExit(
        "Could not find ControlSift. Attach the Kaggle dataset zip from "
        "scripts/package_for_kaggle.py or set REPO_URL to a public clone URL."
    )

os.chdir(work)
print("WORKDIR =", work.resolve())
assert (work / "data" / "processed" / "train.jsonl").exists(), "missing train.jsonl"
assert (work / "scripts" / "run_gemma_baseline.py").exists(), "missing scripts"

In [ ]:
# === Auth via platform secrets only (token never printed) ===
import os

def load_hf_token() -> str:
    # Kaggle
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    # Colab
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    # Env var fallback for advanced users (still do not hardcode)
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if token:
        return token
    raise SystemExit(
        "HF_TOKEN not found. Add it under Kaggle/Colab Secrets. "
        "Do not paste the token into this notebook."
    )

token = load_hf_token()
os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token

from huggingface_hub import login
login(token=token, add_to_git_credential=False)

# Prove login without exposing secret material
print("HF login OK; token length =", len(token), "(value not shown)")
del token

In [ ]:
# === Install GPU deps + package ===
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU — free run will be too slow; enable GPU accelerator.")

!pip -q install -U pip
!pip -q install -e ".[gpu]"
print("Install complete")

In [ ]:
# === Gemma baselines ===
limit_flag = f"--limit {SMOKE_LIMIT}" if SMOKE else ""
splits = ["validation"] if SMOKE else ["validation", "test", "challenge"]

for mode in ["zero_shot", "few_shot"]:
    for split in splits:
        cmd = f"python scripts/run_gemma_baseline.py --mode {mode} --split {split} {limit_flag}"
        print("\n>>", cmd)
        ok = os.system(cmd)
        if ok != 0:
            raise SystemExit(f"Baseline failed: {cmd}")

In [ ]:
# === QLoRA train ===
train_cmd = "python -m controlsift.training.train --smoke" if SMOKE else "python -m controlsift.training.train"
print(">>", train_cmd)
ok = os.system(train_cmd)
if ok != 0:
    raise SystemExit("Training failed")

In [ ]:
# === QLoRA eval ===
limit_flag = f"--limit {SMOKE_LIMIT}" if SMOKE else ""
splits = ["validation"] if SMOKE else ["validation", "test", "challenge"]
for split in splits:
    cmd = f"python scripts/run_gemma_qlora_eval.py --split {split} {limit_flag}"
    print("\n>>", cmd)
    ok = os.system(cmd)
    if ok != 0:
        raise SystemExit(f"QLoRA eval failed: {cmd}")

In [ ]:
# === Package downloadable outputs (metrics/preds/meta; skip huge weights by default) ===
import zipfile
from pathlib import Path

INCLUDE_LARGE_ADAPTER = False  # set True only if you knowingly want a large zip

out_zip = Path("/kaggle/working/controlsift_gpu_outputs.zip")
if not out_zip.parent.exists():
    out_zip = Path("controlsift_gpu_outputs.zip")

roots = [
    Path("results/gemma_zero_shot"),
    Path("results/gemma_few_shot"),
    Path("results/gemma_qlora"),
]

written = []
with zipfile.ZipFile(out_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root in roots:
        if not root.exists():
            continue
        for path in root.rglob("*"):
            if not path.is_file():
                continue
            rel = str(path).replace("\\", "/")
            if path.suffix in {".safetensors", ".bin", ".pt", ".pth"} and not INCLUDE_LARGE_ADAPTER:
                continue
            if "adapter/" in rel and path.suffix not in {".json", ".txt", ".md"} and not INCLUDE_LARGE_ADAPTER:
                # keep tokenizer/config json; skip weight shards
                if path.name.startswith("adapter_model") or path.name.startswith("model"):
                    continue
            zf.write(path, arcname=rel)
            written.append(rel)

print("Wrote", out_zip, "files:", len(written))
print("Download this zip → unzip into your local repo → run:")
print("  python scripts/run_evaluation.py")
print("  python scripts/build_figures.py")

## After download (on your PC)

1. Unzip into the ControlSift repo root (so `results/gemma_*` merge in).
2. `python scripts/run_evaluation.py && python scripts/build_figures.py`
3. Refresh the local site — Gemma rows should leave `pending`.
4. If this was a smoke run only, re-run on Kaggle with `SMOKE = False` when you have quota.

Do not commit HF tokens. Prefer committing metrics/predictions JSON only.